# Project Title & Domain

## Telecom Churn Prediction & Customer Lifetime Value (CLV) Analysis
**Domain:** Telecommunications, customer analytics and business intelligence.

# Project Overview

This project examines customer records from a telecommunications provider to understand customer churn, identify customer groups associated with higher attrition and estimate a revenue-based Customer Lifetime Value (CLV) proxy. Later stages will use Excel, Python, MySQL and Power BI to support retention reporting.

# Business Problem

Customer churn reduces recurring revenue. Telecom teams need to understand how churn varies with contract types, tenure, service choices and monthly charges. The project will investigate these patterns and produce analysis that supports prioritizing retention efforts.

# Project Objectives

1. Measure the overall churn percentage and compare churn percentages across contract types and internet services.
2. Identify at least three customer characteristics associated with churn, using descriptive statistics and charts.
3. Estimate monthly revenue and average revenue per user (ARPU), stating the assumptions used.
4. Develop a transparent, revenue-based CLV proxy and compare customer segments by their estimated value.
5. In later phases, train and evaluate a churn classification model using precision, recall and F1-score.
6. Present key customer segments and churn KPIs in Power BI in later phases.

# Business Value & Use Case

Intended stakeholders include telecom customer success teams, marketing managers and business analysts. The findings will help them understand churn patterns, assess revenue exposure and identify groups to prioritize for review. The dataset does not demonstrate the effectiveness of any retention intervention.

# Dataset Information

**Dataset:** Telco Customer Churn (IBM sample dataset, distributed on Kaggle).

**Original approval link:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**File:** `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` (CSV)

**Size:** 7,043 rows and 21 columns. Each row represents one customer record.

**Key fields:** `customerID`, `tenure`, `Contract`, `InternetService`, `TechSupport`, `MonthlyCharges`, `TotalCharges`, `Churn`.

**Known initial data-type issue:** `TotalCharges` is read as text and includes 11 blank-string entries. This will be handled in later cleaning stages.

# Planned Analysis

In subsequent phases, inspect data quality and duplicates, convert `TotalCharges` to numeric, document missing-value treatment, explore churn by customer segment, analyse tenure and charges, calculate churn and revenue KPIs, derive a clearly labelled CLV proxy, compare classification algorithms, evaluate model results, use MySQL for reporting queries and build an interactive Power BI dashboard. Proposed charts include churn by contract, churn by internet service, tenure distributions and a customer-value segment chart. No outcomes are assumed in advance.

# Scope & Limitations

This project uses the provided 7,043-row customer-level sample, not live subscriber records. The data does not provide actual retention campaign outcomes, customer acquisition costs, profit margins or verified future customer lifetimes. CLV will therefore be an **estimated revenue proxy**, not observed profit-based lifetime value. The `Churn` label represents churn status in the sample; it does not directly verify a next-month event. Analyses describe patterns and associations, not proven causes.

# Phase 1: Load and Inspect the Approved Dataset
The cells below load the unmodified CSV from a relative path and display basic structural checks. Detailed cleaning, EDA and charts are reserved for later phases.

In [1]:
from pathlib import Path
import pandas as pd

dataset_path = Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(dataset_path)
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Dataset shape (rows, columns):", df.shape)

Dataset shape (rows, columns): (7043, 21)


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
print("Data types:\n", df.dtypes)
print("Blank TotalCharges entries:", df["TotalCharges"].astype(str).str.strip().eq("").sum())

Data types:
 customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object
Blank TotalCharges entries: 11


## Initial Observations
The file contains **7,043 rows and 21 columns**, with customer demographic, account, service and billing fields, plus the binary `Churn` field. `TotalCharges` is initially stored as an object/text column rather than a numeric column, and 11 records contain blank strings in this field. These observations guide the planned cleaning work; no transformations are performed in Phase 1.

# Phase 2: Data Preprocessing and Cleaning

## Objective
Assess data quality, handle hidden missing values, standardise data types, encode categorical features, create a scaling demonstration, validate outputs and export datasets. The original CSV remains unchanged.

In [6]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# Preserve the original DataFrame
raw_df = df.copy(deep=True)
df_cleaned = df.copy(deep=True)
print("Working copy created:", df_cleaned.shape)

Working copy created: (7043, 21)


## 2.1 Initial quality checks

In [7]:
print("Shape:", df_cleaned.shape)
print("Missing values before conversion:\n", df_cleaned.isna().sum())
print("Duplicate rows:", df_cleaned.duplicated().sum())
print("Duplicate customer IDs:", df_cleaned["customerID"].duplicated().sum())
print("Whitespace-only TotalCharges:", df_cleaned["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Tenure equals zero:", df_cleaned["tenure"].eq(0).sum())

Shape: (7043, 21)
Missing values before conversion:
 customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
Duplicate rows: 0
Duplicate customer IDs: 0
Whitespace-only TotalCharges: 11
Tenure equals zero: 11


### Observation
The original `TotalCharges` field is read as text. Eleven entries contain whitespace, which ordinary `isna()` does not detect. The data will be converted to numeric, and zero-tenure customers with blank totals will receive zero total charges, consistent with the project SRS.

## 2.2 Standardise text and repair TotalCharges

In [8]:
# Strip outer spaces from text values while preserving missing values
text_cols = df_cleaned.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df_cleaned[col] = df_cleaned[col].astype("string").str.strip()

# Replace blank strings with missing values and convert financial field to numeric
df_cleaned["TotalCharges"] = pd.to_numeric(
    df_cleaned["TotalCharges"].replace("", pd.NA), errors="coerce"
)
print("TotalCharges missing after conversion:", df_cleaned["TotalCharges"].isna().sum())
print("Missing TotalCharges for tenure > 0:",
      df_cleaned.loc[df_cleaned["tenure"] > 0, "TotalCharges"].isna().sum())

# Apply zero only where the account tenure is exactly zero
zero_tenure_missing = df_cleaned["TotalCharges"].isna() & df_cleaned["tenure"].eq(0)
df_cleaned.loc[zero_tenure_missing, "TotalCharges"] = 0.0
print("Imputed zero-tenure rows:", zero_tenure_missing.sum())
print("Remaining missing values:\n", df_cleaned.isna().sum()[lambda s: s.gt(0)])

TotalCharges missing after conversion: 11
Missing TotalCharges for tenure > 0: 0
Imputed zero-tenure rows: 11
Remaining missing values:
 Series([], dtype: int64)


## 2.3 Check duplicates, ranges and categories

In [9]:
print("Duplicate customer IDs:", df_cleaned["customerID"].duplicated().sum())
print("Duplicate complete rows:", df_cleaned.duplicated().sum())
print("Negative tenure:", df_cleaned["tenure"].lt(0).sum())
print("Negative MonthlyCharges:", df_cleaned["MonthlyCharges"].lt(0).sum())
print("Negative TotalCharges:", df_cleaned["TotalCharges"].lt(0).sum())
print("\nNumerical summary:\n", df_cleaned[["tenure", "MonthlyCharges", "TotalCharges"]].describe())

for col in ["Contract", "InternetService", "PaymentMethod", "Churn"]:
    print(f"\n{col} categories:\n{df_cleaned[col].value_counts(dropna=False)}")

Duplicate customer IDs: 0
Duplicate complete rows: 0
Negative tenure: 0
Negative MonthlyCharges: 0
Negative TotalCharges: 0

Numerical summary:
             tenure  MonthlyCharges  TotalCharges
count  7043.000000     7043.000000        7043.0
mean     32.371149       64.761692   2279.734304
std      24.559481       30.090047    2266.79447
min       0.000000       18.250000           0.0
25%       9.000000       35.500000        398.55
50%      29.000000       70.350000       1394.55
75%      55.000000       89.850000        3786.6
max      72.000000      118.750000        8684.8

Contract categories:
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64[pyarrow]

InternetService categories:
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64[pyarrow]

PaymentMethod categories:
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1

### Quality interpretation
Duplicate customer IDs or unexpected categories would require further review. Do not drop records automatically. Check the meaning of anomalies before changing source records.

## 2.4 Create a cleaned dataset

In [10]:
# Retain original categorical labels for EDA and reporting
assert df_cleaned.shape == (7043, 21), "Unexpected source dimensions"
assert df_cleaned["customerID"].is_unique, "Customer IDs are not unique"
assert df_cleaned["TotalCharges"].notna().all(), "TotalCharges still has missing values"
assert df_cleaned["TotalCharges"].ge(0).all(), "Negative total charges detected"
assert df_cleaned["Churn"].isin(["Yes", "No"]).all(), "Unexpected Churn label"
print("Cleaned data validated:", df_cleaned.shape)
df_cleaned.head()

Cleaned data validated: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2.5 Encode binary categories

In [11]:
# Create independent model-oriented data without modifying reporting dataset
df_encoded = df_cleaned.drop(columns=["customerID"]).copy()

# Map only true Yes/No fields. Three-category service fields are handled separately.
yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
for col in yes_no_cols:
    df_encoded[col] = df_encoded[col].map({"No": 0, "Yes": 1}).astype("int64")

# Gender is binary in this sample, but is not coded as Yes/No
df_encoded["gender"] = df_encoded["gender"].map({"Female": 0, "Male": 1}).astype("int64")
print("Binary encoded fields:\n", df_encoded[yes_no_cols + ["gender"]].head())

Binary encoded fields:
    Partner  Dependents  PhoneService  PaperlessBilling  Churn  gender
0        1           0             0                 1      0       0
1        0           0             1                 0      0       1
2        0           0             1                 1      1       1
3        0           0             0                 0      0       1
4        0           0             1                 1      1       0


## 2.6 One-hot encode multi-category features

One-hot encoding avoids imposing an artificial numerical order on contract, internet, payment and service categories. For example, `No internet service` is different from `No`.

In [12]:
# Encode remaining categorical fields; keep all categories for traceability
remaining_categorical = df_encoded.select_dtypes(include=["object", "string", "category"]).columns.tolist()
df_encoded = pd.get_dummies(df_encoded, columns=remaining_categorical, dtype=int)
print("Encoded dataset shape:", df_encoded.shape)
print("Remaining text columns:", df_encoded.select_dtypes(include=["object", "string"]).columns.tolist())
assert not df_encoded.isna().any().any(), "Encoded data has missing values"
assert "Churn" in df_encoded.columns, "Target variable missing"

Encoded dataset shape: (7043, 41)
Remaining text columns: []


## 2.7 Standardise numerical features (demonstration)

For Phase 2 practice, standardise tenure and monthly charges in a separate example DataFrame. **Do not use these full-data-fitted scaling values in a future churn model evaluation.** In the modelling phase, split into training and test data first, then fit `StandardScaler` only on the training set.

In [13]:
# Scaling example for data preparation review only
scaled_preview = df_encoded.copy()
scale_cols = ["tenure", "MonthlyCharges"]
scaler_demo = StandardScaler()
scaled_preview[scale_cols] = scaler_demo.fit_transform(scaled_preview[scale_cols])
print("Scaled feature summary (demonstration only):\n", scaled_preview[scale_cols].describe().round(3))

# Keep exported encoded data unscaled to prevent accidental modelling leakage
print("Exported encoded dataset remains unscaled.")

Scaled feature summary (demonstration only):
          tenure  MonthlyCharges
count  7043.000        7043.000
mean     -0.000          -0.000
std       1.000           1.000
min      -1.318          -1.546
25%      -0.952          -0.973
50%      -0.137           0.186
75%       0.921           0.834
max       1.614           1.794
Exported encoded dataset remains unscaled.


## 2.8 Validate and export processed files

In [14]:
from pathlib import Path

# Supports running notebook from the repository root or the scripts/ directory
project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

cleaned_path = processed_dir / "telco_cleaned.csv"
encoded_path = processed_dir / "telco_encoded.csv"
df_cleaned.to_csv(cleaned_path, index=False)
df_encoded.to_csv(encoded_path, index=False)

# Re-import exported files to confirm files are readable and dimensions are preserved
cleaned_check = pd.read_csv(cleaned_path)
encoded_check = pd.read_csv(encoded_path)
assert cleaned_check.shape == (7043, 21)
assert encoded_check.shape == df_encoded.shape
assert not encoded_check.isna().any().any()
print("Saved:", cleaned_path)
print("Saved:", encoded_path)
print("Cleaned shape:", cleaned_check.shape)
print("Encoded shape:", encoded_check.shape)

Saved: C:\Users\kavya\anaconda3\Entri- Assignments\Telecom_Churn_CLV_Analysis\data\processed\telco_cleaned.csv
Saved: C:\Users\kavya\anaconda3\Entri- Assignments\Telecom_Churn_CLV_Analysis\data\processed\telco_encoded.csv
Cleaned shape: (7043, 21)
Encoded shape: (7043, 41)


## Phase 2 Summary

- Preserved the original CSV and created a separate working copy.
- Identified whitespace-only values in `TotalCharges`.
- Converted `TotalCharges` to numeric and assigned zero only for missing records with zero tenure.
- Validated duplicate identifiers, numeric ranges and categories.
- Kept categorical labels in a reporting-friendly cleaned dataset.
- Encoded binary and multi-category features in a separate dataset.
- Demonstrated standardisation while documenting future train/test leakage precautions.
- Exported `telco_cleaned.csv` and `telco_encoded.csv` for later phases.

Churn prediction, CLV estimation and dashboards are reserved for later phases.

In [15]:
from pathlib import Path

# Create the processed data folder
output_folder = Path("data/processed")
output_folder.mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
df_cleaned.to_csv(
    output_folder / "telco_cleaned.csv",
    index=False
)

# Save the encoded dataset
df_encoded.to_csv(
    output_folder / "telco_encoded.csv",
    index=False
)

print("Both CSV files saved successfully!")
print("Cleaned dataset:", (output_folder / "telco_cleaned.csv").resolve())
print("Encoded dataset:", (output_folder / "telco_encoded.csv").resolve())

Both CSV files saved successfully!
Cleaned dataset: C:\Users\kavya\anaconda3\Entri- Assignments\Telecom_Churn_CLV_Analysis\data\processed\telco_cleaned.csv
Encoded dataset: C:\Users\kavya\anaconda3\Entri- Assignments\Telecom_Churn_CLV_Analysis\data\processed\telco_encoded.csv
